# BrainHack Fall 2026 — OpenScope mesoscope movie + all-ROI peri-stimulus response

**Dataset:** OpenScope Community Predictive Processing, mesoscope two-photon calcium imaging, **DANDI:001768**.

This repaired Colab notebook is based on the documented DANDI:001768 NWB hierarchy:

- each imaging plane is a processing module;
- `dff_timeseries` contains ΔF/F;
- `image_segmentation` contains 512×512 ROI masks;
- `images` contains summary/projection images;
- synchronized stimulus presentations are stored in NWB `TimeIntervals`.

## What this notebook produces

1. A **60-second ROI activity movie**. This reconstructs activity by putting each ROI's ΔF/F value into its spatial footprint. It does **not** claim to be the raw ScanImage movie.
2. A **peri-stimulus calcium response for every ROI** in the selected plane.
3. The **grand population mean ± SEM across all ROIs**.
4. An **all-ROI heat map**.

The code contains explicit structural checks and supports both `image_mask` and `pixel_mask` representations rather than silently creating a 1×1/blank movie.

In [ ]:
!pip -q install "dandi>=0.70" "pynwb>=2.8" h5py remfile scipy matplotlib pandas imageio imageio-ffmpeg tqdm

In [ ]:
import random, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from tqdm.auto import tqdm

from dandi.dandiapi import DandiAPIClient
import remfile, h5py
from pynwb import NWBHDF5IO

warnings.filterwarnings("ignore")

DANDISET_ID = "001768"
DANDISET_VERSION = "draft"
RANDOM_SEED = 2026

MOVIE_SECONDS = 60
MOVIE_FPS = 10
PRE = 2.0
POST = 5.0
DT = 0.1

# None = all eligible stimulus presentations.
# Set to an integer such as 300 only for a quicker test.
MAX_TRIALS = None

print("Configuration loaded.")

## 1 — Select and remotely open one mesoscope NWB

The random selection is reproducible. The NWB is streamed with HTTP range requests rather than downloading the entire asset.

In [ ]:
rng = random.Random(RANDOM_SEED)
client = DandiAPIClient()
dandiset = client.get_dandiset(DANDISET_ID, version_id=DANDISET_VERSION)

assets = [a for a in dandiset.get_assets() if a.path.lower().endswith(".nwb")]
if not assets:
    raise RuntimeError("No .nwb assets were found in DANDI:001768.")

asset = rng.choice(assets)
print("Selected NWB:", asset.path)
print("Asset size (GB):", round(asset.size/1e9, 2) if asset.size else "unknown")

url = asset.get_content_url(follow_redirects=1, strip_query=True)
rf = remfile.File(url=url)
hf = h5py.File(rf, "r")
io = NWBHDF5IO(file=hf, mode="r")
nwb = io.read()

print("Session ID:", nwb.session_id)
print("Processing modules:", list(nwb.processing.keys()))

## 2 — Select a real mesoscope imaging plane

The documented mesoscope hierarchy uses `dff_timeseries` directly in each plane module. This cell first looks for that exact interface and only then uses a conservative fallback.

In [ ]:
candidates = []

for name, mod in nwb.processing.items():
    names = list(mod.data_interfaces.keys())

    if "dff_timeseries" in names:
        candidates.append((name, mod, mod.data_interfaces["dff_timeseries"]))
    else:
        for interface_name, obj in mod.data_interfaces.items():
            lname = interface_name.lower()
            if ("dff" in lname or "df_f" in lname) and hasattr(obj, "data"):
                candidates.append((name, mod, obj))

if not candidates:
    print("Actual processing structure:")
    for name, mod in nwb.processing.items():
        print("\n", name)
        for interface_name, obj in mod.data_interfaces.items():
            print("  ", interface_name, "->", type(obj).__name__)
    raise RuntimeError("No ΔF/F RoiResponseSeries could be found.")

plane_name, plane_mod, dff_obj = candidates[0]

print("Selected plane:", plane_name)
print("ΔF/F interface:", dff_obj.name)
print("Type:", type(dff_obj).__name__)
print("Shape:", dff_obj.data.shape)
print("\nPlane interfaces:")
for name, obj in plane_mod.data_interfaces.items():
    print(" ", name, "->", type(obj).__name__)

### Normalize the ΔF/F array to time × ROI

NWB `RoiResponseSeries` normally stores time × ROI. We verify that against timestamps rather than assuming it.

In [ ]:
shape = tuple(dff_obj.data.shape)
if len(shape) != 2:
    raise RuntimeError(f"Expected 2-D ΔF/F data, found {shape}.")

if dff_obj.timestamps is not None:
    t = np.asarray(dff_obj.timestamps[:], dtype=float)
else:
    if dff_obj.rate is None:
        raise RuntimeError("ΔF/F series has neither timestamps nor a sampling rate.")
    t = float(dff_obj.starting_time) + np.arange(shape[0]) / float(dff_obj.rate)

if len(t) == shape[0]:
    time_axis = 0
    n_roi = shape[1]
elif len(t) == shape[1]:
    time_axis = 1
    n_roi = shape[0]
else:
    raise RuntimeError(
        f"Timestamp count ({len(t)}) matches neither ΔF/F dimension {shape}."
    )

if len(t) < 2:
    raise RuntimeError("Too few calcium timestamps.")

print("Time samples:", len(t))
print("ROIs in ΔF/F:", n_roi)
print("Approximate sampling rate:", round(1/np.median(np.diff(t)), 3), "Hz")

## 3 — Extract the actual ROI spatial footprints

This is the part that fixes the blank-movie error. We enter the plane's `ImageSegmentation`, obtain a `PlaneSegmentation`, inspect its real columns, and support either:

- dense `image_mask` arrays, or
- weighted NWB `pixel_mask` arrays.

The notebook refuses to continue if it recovers no spatial data.

In [ ]:
# Find ImageSegmentation in the selected plane.
image_seg = None
image_seg_name = None

for name, obj in plane_mod.data_interfaces.items():
    if type(obj).__name__ == "ImageSegmentation":
        image_seg = obj
        image_seg_name = name
        break

if image_seg is None:
    raise RuntimeError(
        f"No ImageSegmentation interface was found in plane {plane_name}. "
        f"Interfaces: {list(plane_mod.data_interfaces.keys())}"
    )

print("ImageSegmentation:", image_seg_name)
print("Plane segmentations:", list(image_seg.plane_segmentations.keys()))

if not image_seg.plane_segmentations:
    raise RuntimeError("ImageSegmentation contains no PlaneSegmentation tables.")

# Prefer the segmentation referenced by the dF/F series when possible.
seg = None
try:
    seg = dff_obj.rois.table
except Exception:
    pass

if seg is None:
    seg = next(iter(image_seg.plane_segmentations.values()))

print("Using PlaneSegmentation:", seg.name)
print("Segmentation ROIs:", len(seg))
print("Columns:", list(seg.colnames))

In [ ]:
def get_dense_roi_masks(seg):
    colnames = list(seg.colnames)

    # Dense image masks
    if "image_mask" in colnames:
        masks = []
        expected_shape = None
        for i in tqdm(range(len(seg)), desc="Reading image masks"):
            m = seg["image_mask"][i]
            if m is None:
                masks.append(None)
                continue
            m = np.asarray(m, dtype=np.float32)
            if m.ndim != 2 or m.size == 0:
                masks.append(None)
                continue
            if expected_shape is None:
                expected_shape = m.shape
            if m.shape != expected_shape:
                masks.append(None)
                continue
            masks.append(m)

        valid = [m for m in masks if m is not None and np.any(np.isfinite(m) & (m != 0))]
        if valid:
            return masks, valid[0].shape, "image_mask"

    # Sparse weighted pixel masks: rows are x, y, weight.
    if "pixel_mask" in colnames:
        sparse = []
        max_x = max_y = -1

        for i in tqdm(range(len(seg)), desc="Reading pixel masks"):
            pm = seg["pixel_mask"][i]
            if pm is None:
                sparse.append(None)
                continue

            pm = np.asarray(pm)
            if pm.ndim != 2 or pm.shape[1] < 3 or len(pm) == 0:
                sparse.append(None)
                continue

            x = pm[:, 0].astype(int)
            y = pm[:, 1].astype(int)
            w = pm[:, 2].astype(np.float32)

            good = (x >= 0) & (y >= 0) & np.isfinite(w)
            x, y, w = x[good], y[good], w[good]

            if len(x) == 0:
                sparse.append(None)
                continue

            max_x = max(max_x, int(x.max()))
            max_y = max(max_y, int(y.max()))
            sparse.append((x, y, w))

        if max_x < 0 or max_y < 0:
            raise RuntimeError("pixel_mask exists but no valid pixel coordinates were recovered.")

        # Mesoscope documentation specifies 512×512 masks. Never shrink below
        # the observed extent; use 512×512 when the coordinates fit it.
        W = 512 if max_x < 512 else max_x + 1
        H = 512 if max_y < 512 else max_y + 1

        masks = []
        for item in sparse:
            if item is None:
                masks.append(None)
                continue
            x, y, w = item
            m = np.zeros((H, W), dtype=np.float32)
            m[y, x] = w
            masks.append(m)

        return masks, (H, W), "pixel_mask"

    raise RuntimeError(
        "PlaneSegmentation has neither usable image_mask nor pixel_mask. "
        f"Available columns: {colnames}"
    )

roi_masks, (H, W), mask_type = get_dense_roi_masks(seg)

n_use = min(n_roi, len(roi_masks))
roi_masks = roi_masks[:n_use]

valid_count = sum(
    m is not None and np.any(np.isfinite(m) & (m != 0))
    for m in roi_masks
)

print("\nMask representation:", mask_type)
print("Movie field dimensions:", H, "x", W)
print("ROIs in dF/F:", n_roi)
print("Masks in segmentation:", len(seg))
print("ROIs used:", n_use)
print("Valid spatial masks:", valid_count)

if H <= 1 or W <= 1 or valid_count == 0:
    raise RuntimeError(
        "Spatial mask extraction failed; refusing to render a blank/1×1 movie."
    )

### Spatial sanity check

**Do not skip this diagnostic.** You should see a 2-D field containing many ROI footprints. If the field is blank, the notebook intentionally stops before movie rendering.

In [ ]:
roi_map = np.zeros((H, W), dtype=np.float32)

for m in roi_masks:
    if m is not None:
        roi_map += np.nan_to_num(m, nan=0.0)

if not np.any(roi_map != 0):
    raise RuntimeError("ROI sanity-check image is empty.")

plt.figure(figsize=(7, 7))
plt.imshow(roi_map, cmap="gray", origin="upper")
plt.title(f"{plane_name}: {valid_count} recovered ROI footprints")
plt.xlabel("x pixel")
plt.ylabel("y pixel")
plt.colorbar(label="summed ROI mask weight")
plt.tight_layout()
plt.show()

## 4 — Find synchronized stimulus onsets

The mesoscope NWBs store synchronized stimulus tables as `TimeIntervals`. We choose a non-empty interval table containing `start_time`, report exactly which table is used, and use those onsets for both the movie annotation and PSTH-style analysis.

In [ ]:
tables = []

for name, tab in nwb.intervals.items():
    try:
        df = tab.to_dataframe()
    except Exception:
        continue

    if "start_time" in df.columns and len(df) > 0:
        on = pd.to_numeric(df["start_time"], errors="coerce").to_numpy(dtype=float)
        on = on[np.isfinite(on)]
        if len(on):
            tables.append((name, df, on))

if not tables and nwb.trials is not None:
    df = nwb.trials.to_dataframe()
    if "start_time" in df.columns:
        on = pd.to_numeric(df["start_time"], errors="coerce").to_numpy(dtype=float)
        on = on[np.isfinite(on)]
        if len(on):
            tables.append(("trials", df, on))

if not tables:
    raise RuntimeError("No stimulus/trial TimeIntervals table with start_time was found.")

print("Candidate interval tables:")
for name, df, on in tables:
    print(f"  {name}: {len(on)} onsets; columns={list(df.columns)[:8]}")

# Use the largest stimulus-like table; this is more robust than blindly taking
# the first interval table.
tables.sort(key=lambda x: len(x[2]), reverse=True)
stim_name, stim_df, stim_onsets = tables[0]
stim_onsets = np.sort(stim_onsets)

print("\nSelected stimulus table:", stim_name)
print("Number of stimulus onsets:", len(stim_onsets))
display(stim_df.head())

## 5 — Build a 60-second activity movie around stimulus presentations

The movie is a **processed ROI-activity reconstruction**: each ROI footprint is filled with its contemporaneous ΔF/F. Frames containing a stimulus onset are labeled.

To keep Colab responsive, the movie is rendered at 10 fps by sampling the calcium time series at the nearest available timestamps.

In [ ]:
# Choose a 60-second interval containing a central stimulus.
center = float(stim_onsets[len(stim_onsets)//2])
movie_start = max(float(t[0]), center - 20.0)
movie_end = min(float(t[-1]), movie_start + MOVIE_SECONDS)

if movie_end - movie_start < MOVIE_SECONDS and t[-1] - t[0] >= MOVIE_SECONDS:
    movie_start = float(movie_end - MOVIE_SECONDS)

duration = movie_end - movie_start
if duration <= 0:
    raise RuntimeError("Could not construct a valid movie interval.")

sample_times = np.arange(movie_start, movie_end, 1.0/MOVIE_FPS)
sample_idx = np.searchsorted(t, sample_times)
sample_idx = np.clip(sample_idx, 0, len(t)-1)

idx0 = int(sample_idx.min())
idx1 = int(sample_idx.max()) + 1

if time_axis == 0:
    dff_block = np.asarray(dff_obj.data[idx0:idx1, :n_use], dtype=np.float32)
else:
    dff_block = np.asarray(dff_obj.data[:n_use, idx0:idx1], dtype=np.float32).T

movie_values = dff_block[sample_idx - idx0, :]

finite = movie_values[np.isfinite(movie_values)]
if finite.size == 0:
    raise RuntimeError("Movie ΔF/F block contains no finite values.")

vmin, vmax = np.nanpercentile(finite, [5, 99])
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
    vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
if vmax <= vmin:
    vmax = vmin + 1e-6

in_window = stim_onsets[(stim_onsets >= movie_start) & (stim_onsets < movie_end)]

print(f"Movie interval: {movie_start:.3f}–{movie_end:.3f} s ({duration:.1f} s)")
print("Frames:", len(sample_times))
print("Stimulus onsets in movie:", len(in_window))
print("ΔF/F display range:", vmin, "to", vmax)

In [ ]:
out_mp4 = "/content/openscope_60s_roi_activity.mp4"

writer = imageio.get_writer(
    out_mp4,
    fps=MOVIE_FPS,
    codec="libx264",
    quality=7,
    macro_block_size=None
)

fig, ax = plt.subplots(figsize=(7, 7), dpi=100)
canvas = np.full((H, W), np.nan, dtype=np.float32)

im = ax.imshow(
    canvas,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    origin="upper",
    interpolation="nearest"
)
ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
title = ax.set_title("")
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("ΔF/F")

for tm, vals in tqdm(
    zip(sample_times, movie_values),
    total=len(sample_times),
    desc="Rendering 60-s movie"
):
    numerator = np.zeros((H, W), dtype=np.float32)
    denominator = np.zeros((H, W), dtype=np.float32)

    for r, m in enumerate(roi_masks):
        if m is None or r >= len(vals) or not np.isfinite(vals[r]):
            continue

        mm = np.nan_to_num(m, nan=0.0)
        positive = mm != 0
        numerator[positive] += vals[r] * mm[positive]
        denominator[positive] += np.abs(mm[positive])

    occupied = denominator > 0
    canvas.fill(np.nan)
    canvas[occupied] = numerator[occupied] / denominator[occupied]
    im.set_data(canvas)

    # A frame represents a 1/FPS-wide display interval.
    is_stim = np.any(np.abs(in_window - tm) <= 0.5/MOVIE_FPS)
    flag = " | STIMULUS ONSET" if is_stim else ""
    title.set_text(f"{plane_name} | elapsed {tm-movie_start:5.1f} s{flag}")

    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[..., :3]
    writer.append_data(frame)

writer.close()
plt.close(fig)

if not os.path.exists(out_mp4) or os.path.getsize(out_mp4) == 0:
    raise RuntimeError("Movie file was not created correctly.")

print("Movie saved:", out_mp4)
print("Movie size (MB):", round(os.path.getsize(out_mp4)/1e6, 2))

In [ ]:
from IPython.display import Video, display
display(Video(out_mp4, embed=True))

# Optional Colab download:
# from google.colab import files
# files.download(out_mp4)

## 6 — Full peri-stimulus calcium response for all ROIs

A true PSTH counts spikes. For calcium imaging the analogous quantity is a **peri-stimulus ΔF/F time course**.

For every eligible stimulus presentation, this section:

1. samples all ROIs onto a common −2 to +5 s grid;
2. baseline-corrects each ROI/trial using the pre-stimulus period;
3. averages across trials separately for every ROI;
4. computes the grand population mean and SEM across ROIs.

No subset of neurons is selected for the population result.

In [ ]:
rel_t = np.arange(-PRE, POST + DT/2, DT)

eligible = stim_onsets[
    (stim_onsets + rel_t[0] >= t[0]) &
    (stim_onsets + rel_t[-1] <= t[-1])
]

if len(eligible) == 0:
    raise RuntimeError("No stimulus onsets have a complete peri-stimulus window.")

if MAX_TRIALS is not None and len(eligible) > MAX_TRIALS:
    rr = np.random.default_rng(RANDOM_SEED)
    eligible = np.sort(rr.choice(eligible, MAX_TRIALS, replace=False))

print("Eligible stimulus presentations:", len(eligible))
print("All ROIs used:", n_use)
print("Relative time bins:", len(rel_t))

In [ ]:
# Read the single contiguous ΔF/F range covering all eligible trials.
lo = max(0, np.searchsorted(t, eligible.min() + rel_t[0]) - 1)
hi = min(len(t), np.searchsorted(t, eligible.max() + rel_t[-1]) + 2)

tt = t[lo:hi]

if time_axis == 0:
    all_dff = np.asarray(dff_obj.data[lo:hi, :n_use], dtype=np.float32)
else:
    all_dff = np.asarray(dff_obj.data[:n_use, lo:hi], dtype=np.float32).T

roi_sum = np.zeros((n_use, len(rel_t)), dtype=np.float64)
roi_count = np.zeros((n_use, len(rel_t)), dtype=np.int64)

baseline_bins = rel_t < 0

for onset in tqdm(eligible, desc="Aligning all ROIs to stimuli"):
    q = onset + rel_t
    right = np.searchsorted(tt, q)
    right = np.clip(right, 1, len(tt)-1)
    left = right - 1

    denom = tt[right] - tt[left]
    denom[denom == 0] = np.nan
    frac = ((q - tt[left]) / denom).astype(np.float32)

    # shape: relative-time × ROI
    Y = (
        all_dff[left, :] * (1.0 - frac[:, None]) +
        all_dff[right, :] * frac[:, None]
    )

    baseline = np.nanmean(Y[baseline_bins, :], axis=0)
    Y = Y - baseline[None, :]

    valid = np.isfinite(Y)
    roi_sum += np.where(valid, Y, 0.0).T
    roi_count += valid.T

roi_psth = np.divide(
    roi_sum,
    roi_count,
    out=np.full(roi_sum.shape, np.nan, dtype=np.float64),
    where=roi_count > 0
)

valid_rois = np.any(np.isfinite(roi_psth), axis=1)
if not np.any(valid_rois):
    raise RuntimeError("No finite peri-stimulus ROI responses were computed.")

roi_psth_valid = roi_psth[valid_rois]
pop_mean = np.nanmean(roi_psth_valid, axis=0)
n_eff = np.sum(np.isfinite(roi_psth_valid), axis=0)
pop_sem = (
    np.nanstd(roi_psth_valid, axis=0, ddof=1) /
    np.sqrt(np.maximum(n_eff, 1))
)

print("ROIs with valid peri-stimulus responses:", roi_psth_valid.shape[0])

In [ ]:
plt.figure(figsize=(10, 6))

for trace in roi_psth_valid:
    plt.plot(rel_t, trace, alpha=0.08, linewidth=0.6)

plt.fill_between(
    rel_t,
    pop_mean-pop_sem,
    pop_mean+pop_sem,
    alpha=0.25,
    label="SEM across ROIs"
)
plt.plot(
    rel_t,
    pop_mean,
    linewidth=2.5,
    label=f"Population mean (n={roi_psth_valid.shape[0]} ROIs)"
)
plt.axvline(0, linestyle="--", linewidth=1.5, label="Stimulus onset")
plt.axhline(0, linewidth=0.7)
plt.xlabel("Time from stimulus onset (s)")
plt.ylabel("Baseline-corrected ΔF/F")
plt.title(f"{plane_name}: all-ROI peri-stimulus calcium response")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Heat map of every ROI's trial-averaged response.
response_window = (rel_t >= 0) & (rel_t <= 1)
score = np.nanmean(roi_psth_valid[:, response_window], axis=1)
order = np.argsort(np.nan_to_num(score, nan=-np.inf))

finite_heat = roi_psth_valid[np.isfinite(roi_psth_valid)]
lo_c, hi_c = np.nanpercentile(finite_heat, [2, 98]) if finite_heat.size else (-1, 1)

plt.figure(figsize=(10, 8))
plt.imshow(
    roi_psth_valid[order],
    aspect="auto",
    extent=[rel_t[0], rel_t[-1], roi_psth_valid.shape[0], 0],
    cmap="viridis",
    vmin=lo_c,
    vmax=hi_c
)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Time from stimulus onset (s)")
plt.ylabel("ROI (sorted by 0–1 s response)")
plt.title(f"{plane_name}: trial-averaged response of every ROI")
plt.colorbar(label="Baseline-corrected ΔF/F")
plt.tight_layout()
plt.show()

## 7 — Save numerical results

This makes the population response and every ROI's average response easy to reuse in later BrainHack analyses.

In [ ]:
population_df = pd.DataFrame({
    "time_from_stimulus_s": rel_t,
    "population_mean_dff": pop_mean,
    "population_sem_dff": pop_sem,
    "n_rois": n_eff
})
population_csv = "/content/openscope_population_peristimulus.csv"
population_df.to_csv(population_csv, index=False)

roi_df = pd.DataFrame(
    roi_psth_valid.T,
    index=rel_t,
    columns=[f"ROI_{i}" for i in np.flatnonzero(valid_rois)]
)
roi_df.index.name = "time_from_stimulus_s"
roi_csv = "/content/openscope_all_roi_peristimulus.csv"
roi_df.to_csv(roi_csv)

print("Saved:")
print(" ", population_csv)
print(" ", roi_csv)
print(" ", out_mp4)

## Interpretation

The thick trace is the mean response of the complete valid ROI population in this imaging plane; the faint traces and heat map show the heterogeneity hidden by that mean.

The next predictive-processing analysis should split the synchronized stimulus table into meaningful conditions (for example recurring/control versus deviant/mismatch/omission) rather than pooling all stimulus presentations.

### Project/data references

- OpenScope Community Predictive Processing: https://allenneuraldynamics.github.io/openscope-community-predictive-processing/
- Current project data-release/NWB map: https://allenneuraldynamics.github.io/openscope_p3_data_release_paper/
- DANDI mesoscope Dandiset 001768: https://dandiarchive.org/dandiset/001768
- Neurodata Without Borders: https://www.nwb.org/

In [ ]:
# Close the remote handles after all analyses are finished.
try:
    io.close()
except Exception:
    pass
try:
    hf.close()
except Exception:
    pass
try:
    rf.close()
except Exception:
    pass
try:
    client.close()
except Exception:
    pass

print("Remote resources closed.")